### 11: отбор признаков


In [ ]:
from sklearn.feature_selection import SelectFromModel, SelectKBest, mutual_info_classif

In [ ]:
df = df_main_with_weather.copy()
df = df.loc[:, ~df.columns.duplicated()]

df['RECDATE'] = pd.to_datetime(df['RECDATE'])
df = df.sort_values(['Master Site', 'RECDATE']).reset_index(drop=True)
df['target_today'] = (df['Cell Avail Base Tech (%)'] < 99.8).astype(int)
df['target_next_day'] = df.groupby('Master Site')['target_today'].shift(-1)
df = df.dropna(subset=['target_next_day'])
df['target_next_day'] = df['target_next_day'].astype(int)

tech_cols = ['Cell Avail 2G (%)', 'Cell Avail 3G (%)', 'Cell Avail 4G (%)', 'Cell Avail Base Tech (%)']
weather_cols = ['TempDelta', 'MaxWind', 'MaxInt', 'Rain', 'Snow', 'Sprinkling', 'Hail', 'U', 'Po']
all_feature_names = tech_cols + weather_cols
feature_cols = [c for c in all_feature_names if c in df.columns]

split_date = df['RECDATE'].quantile(0.8)
train_df = df[df['RECDATE'] <= split_date].copy()
test_df  = df[df['RECDATE'] >  split_date].copy()

X_train = train_df[feature_cols].fillna(0)
y_train = train_df['target_next_day']
X_test  = test_df[feature_cols].fillna(0)
y_test  = test_df['target_next_day']

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

In [ ]:
lasso_selector = SelectFromModel(LogisticRegression(
    penalty='l1',
    solver='liblinear',
    random_state=42,
    class_weight='balanced'
))
lasso_selector.fit(X_train_scaled, y_train)

selected_features_lr = X_train_scaled.columns[lasso_selector.get_support()].tolist()
print(f"Отобрано признаков ({len(selected_features_lr)} из {len(feature_cols)}): \n{selected_features_lr}")

X_train_lr = X_train_scaled[selected_features_lr]
X_test_lr = X_test_scaled[selected_features_lr]

# Обучение финальной модели на отобранных признаках
model_lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    solver='lbfgs'
)
model_lr.fit(X_train_lr, y_train)

y_pred_lr = model_lr.predict(X_test_lr)
y_proba_lr = model_lr.predict_proba(X_test_lr)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.4f}")
print(classification_report(y_test, y_pred_lr, target_names=['OK (>=99.8%)', 'Problem (<99.8%)'], digits=4))

coef_df = pd.DataFrame({'feature': selected_features_lr, 'coef': model_lr.coef_[0]}).sort_values('coef', key=abs, ascending=False)
print("Топ признаков (по весам в финальной модели):")
print(coef_df.head(10).to_string(index=False))

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'
)
rf_base.fit(X_train, y_train)

# Отбираем признаки (чья важность выше средней)
rf_selector = SelectFromModel(rf_base, prefit=True)
selected_features_rf = X_train.columns[rf_selector.get_support()].tolist()
print(f"Отобрано признаков ({len(selected_features_rf)} из {len(feature_cols)}): \n{selected_features_rf}")

X_train_rf = X_train[selected_features_rf]
X_test_rf = X_test[selected_features_rf]

rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced'
)
rf_model.fit(X_train_rf, y_train)

y_pred_rf   = rf_model.predict(X_test_rf)
y_proba_rf  = rf_model.predict_proba(X_test_rf)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_rf):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['OK (>=99.8%)', 'Problem (<99.8%)'], digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

# Важность для отобранных признаков
df_imp = pd.DataFrame({
    'Feature': selected_features_rf,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
print(df_imp.head(15).to_string(index=False))

plt.figure(figsize=(10, 8))
top_n = min(15, len(selected_features_rf))
sns.barplot(x='Importance', y='Feature', data=df_imp.head(top_n), palette='viridis')
plt.title(f'Топ {top_n} признаков (Random Forest с отбором)', fontsize=14)
plt.xlabel('Gini Importance', fontsize=12)
plt.ylabel('Признак', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.naive_bayes import GaussianNB

k_best = min(10, len(feature_cols))
mi_selector = SelectKBest(score_func=mutual_info_classif, k=k_best)
mi_selector.fit(X_train, y_train)

selected_features_nb = X_train.columns[mi_selector.get_support()].tolist()
print(f"Отобрано {k_best} лучших признаков по взиамной информации: \n{selected_features_nb}")

X_train_nb = X_train[selected_features_nb]
X_test_nb = X_test[selected_features_nb]

model_nb = GaussianNB()
model_nb.fit(X_train_nb, y_train)

y_pred_nb = model_nb.predict(X_test_nb)
y_proba_nb = model_nb.predict_proba(X_test_nb)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_nb):.4f}")
print("Матрица ошибок:")
print(confusion_matrix(y_test, y_pred_nb))
print("Отчёт по классам:")
print(classification_report(y_test, y_pred_nb, target_names=['Нет аварии', 'Авария']))